In [1]:
import sys
sys.path.append("../src")

In [2]:
from pathlib import Path
import json
import shutil

import matplotlib.pyplot as plt
from dotenv import load_dotenv
load_dotenv()

from agora.debate_analyzer import DebateAnalyzer
from agora.workflows import (
    build_persona_agent_configs,
    load_debate_construction,
    load_persona_catalog,
    load_question_catalog,
    load_prompt_catalog,
    print_agent_histories,
    run_debate_session,
)

# Import persona evaluation components
from agora.persona_evaluator import (
    PersonaEvaluator,
    plot_persona_adherence,
)
from agora.llm import OpenRouterClient

# Persona debate configuration
Configure participant personas, question, and runtime controls.

In [3]:
# Load the available configurations
prompt_path = Path('../data/prompts.json')
personas_path = Path('../data/personas.json')
questions_path = Path('../data/questions.json')
debate_construction_path = Path('../data/debate_construction.json')

personas = load_persona_catalog(personas_path)
questions = load_question_catalog(questions_path)
prompt_catalog = load_prompt_catalog(prompt_path)
debate_construction = load_debate_construction(debate_construction_path)

## Run Experiment Function (with Persona Evaluation)

In [4]:
def run_experiment(
    scenario_id: str,
    question_index: int = 0,
    question_variant: str = "controversial",
    side_order: str = "12",
    debate_arena: str = "neutral",
    alpha_model: str = "anthropic/claude-sonnet-4.5",
    beta_model: str = "anthropic/claude-sonnet-4.5",
    turns_per_agent: int = 10,
    verbose: bool = True,
    show_plots: bool = True,
    evaluate_persona: bool = True,  # NEW: Toggle persona evaluation
    persona_eval_model: str = "anthropic/claude-sonnet-4",  # NEW: Model for evaluation
    persona_eval_verbose: bool = False,  # NEW: Verbose persona evaluation
    persona_n_samples: int = 5,  # NEW: Number of samples for persona eval
):
    """
    Run a complete debate experiment with evaluation.
    
    Args:
        scenario_id: ID from debate_construction["pair_scenarios"]
        question_index: Which question within the scenario (0-based)
        question_variant: "agreeable" or "controversial"
        side_order: "12" (side_1=alpha) or "21" (side_2=alpha)
        debate_arena: "neutral" or "bias" (uses alpha's arena)
        alpha_model: Model for alpha agent
        beta_model: Model for beta agent
        turns_per_agent: Number of debate turns per agent
        verbose: Print debate output
        show_plots: If True, display plots in notebook; if False, just save to file
        evaluate_persona: If True, run persona adherence evaluation
        persona_eval_model: Model to use for persona evaluation
        persona_eval_verbose: If True, print persona evaluation progress
        persona_n_samples: Number of samples to use in persona evaluation
    """
    # === Parse scenario ===
    scenarios = debate_construction.get("pair_scenarios", [])
    scenario = next((s for s in scenarios if s.get("id") == scenario_id), None)
    if scenario is None:
        raise ValueError(f"Scenario '{scenario_id}' not found")
    
    # Scenario has single question_id
    question_id = scenario.get("question_id", "")
    
    # Map sides
    s1, s2 = scenario["side_1_persona_id"], scenario["side_2_persona_id"]
    alpha_persona_id, beta_persona_id = (s1, s2) if side_order == "12" else (s2, s1)
    
    # Arena override (assuming NEUTRAL_DEBATE_ARENA is defined)
    try:
        from agora.workflows import NEUTRAL_DEBATE_ARENA
        debate_arena_override = NEUTRAL_DEBATE_ARENA if debate_arena == "neutral" else None
    except ImportError:
        debate_arena_override = None  # Fallback if not defined
    
    arena_display = "neutral" if debate_arena == "neutral" else f"bias ({alpha_persona_id})"
    
    print(f"[run] scenario={scenario_id}  q={question_id}/{question_variant}  alpha={alpha_persona_id}  beta={beta_persona_id}  arena={arena_display}")
    
    # === Paths ===
    run_name = f'{scenario_id}_{question_id}_{question_variant}_{side_order}_{debate_arena}'
    snapshot_dir = Path('../outputs/snapshots')
    snapshot_dir.mkdir(parents=True, exist_ok=True)
    snapshot_path = snapshot_dir / f'{run_name}.json'
    
    eval_output_dir = Path('../outputs/evaluation_results') / run_name
    eval_output_dir.mkdir(parents=True, exist_ok=True)
    
    # === Run debate ===
    agent_configs = build_persona_agent_configs(
        alpha_persona_id=alpha_persona_id,
        beta_persona_id=beta_persona_id,
        question_id=question_id,
        question_variant=question_variant,
        debate_arena_override=debate_arena_override,
        personas=personas,
        questions=questions,
        alpha_model=alpha_model,
        beta_model=beta_model,
        prompt_set='default',
        prompt_catalog=prompt_catalog,
        private_response_keep=False,
        pre_interview_keep=False,
        post_interview_keep=False,
    )
    
    agora, agents = run_debate_session(
        agent_configs,
        turns_per_agent=turns_per_agent,
        verbose=verbose,
        snapshot_path=snapshot_path,
        load_snapshot_flag=False,
        save_snapshot_flag=True,
        skip_first_agent_first_reflection=True,
    )
    
    # === Evaluation ===
    analyzer = DebateAnalyzer(agora.history())
    intra_scores = analyzer.compute_intra_agent_honesty()
    inter_external = analyzer.compute_inter_agent_alignment("public_speech", "public_speech")
    inter_internal = analyzer.compute_inter_agent_alignment("private_reflection", "private_reflection")
    
    # === NEW: Persona Adherence Evaluation ===
    persona_eval_dict = None
    if evaluate_persona:
        print("\n[Persona Evaluation] Running persona adherence evaluation...")
        
        # Initialize LLM client for evaluation
        llm_client = OpenRouterClient()
        
        # Initialize evaluator
        evaluator = PersonaEvaluator(
            llm_client=llm_client,
            personas=personas,
            model=persona_eval_model,
        )
        
        # Run evaluation
        persona_evaluation = evaluator.evaluate_debate_from_history(
            memory_turns=agora.history(),
            alpha_persona_id=alpha_persona_id,
            beta_persona_id=beta_persona_id,
            verbose=persona_eval_verbose,
            n_samples=persona_n_samples, 
        )
        
        # Convert to dict for saving
        persona_eval_dict = persona_evaluation.to_dict()
        
        print(f"[Persona Evaluation] Alpha final scores - Public: {persona_evaluation.alpha.full_debate_public_score}, Private: {persona_evaluation.alpha.full_debate_private_score}")
        print(f"[Persona Evaluation] Beta final scores - Public: {persona_evaluation.beta.full_debate_public_score}, Private: {persona_evaluation.beta.full_debate_private_score}")
    
    # === Save plots ===
    alpha_name = personas['personas'][alpha_persona_id]['name']
    beta_name = personas['personas'][beta_persona_id]['name']
    label_map = {'Alpha': f'Alpha: {alpha_name}', 'Beta': f'Beta: {beta_name}'}
    
    # Intra-agent plot
    fig1, ax1 = plt.subplots(figsize=(10, 6))
    for speaker, data in intra_scores.items():
        ax1.plot(data["turns"], data["scores"], marker='o', label=label_map.get(speaker, speaker))
    ax1.set_title(f'Intra-Agent Honesty - {question_id} - {question_variant}\nSide: {side_order} | Arena: {debate_arena}')
    ax1.set_xlabel('Debate Turn'); ax1.set_ylabel('Cosine Similarity'); ax1.set_ylim(0, 1)
    ax1.legend(); ax1.grid(); plt.tight_layout()
    fig1.savefig(eval_output_dir / 'intra_agent.png', dpi=150, bbox_inches='tight')
    if show_plots:
        plt.show()
    plt.close(fig1)
    
    # Inter-agent plot
    fig2, ax2 = plt.subplots(figsize=(10, 6))
    ax2.plot(inter_external["turns"], inter_external["scores"], marker='o', label='External (Public)')
    ax2.plot(inter_internal["turns"], inter_internal["scores"], marker='o', label='Internal (Private)')
    ax2.set_title(f'Inter-Agent Alignment - {question_id} - {question_variant}\nAlpha: {alpha_name} | Beta: {beta_name} | Side: {side_order} | Arena: {debate_arena}')
    ax2.set_xlabel('Debate Turn'); ax2.set_ylabel('Cosine Similarity'); ax2.set_ylim(0, 1)
    ax2.legend(); ax2.grid(); plt.tight_layout()
    fig2.savefig(eval_output_dir / 'inter_agent.png', dpi=150, bbox_inches='tight')
    if show_plots:
        plt.show()
    plt.close(fig2)
    
    # NEW: Persona adherence plot
    if persona_eval_dict:
        persona_plot_path = eval_output_dir / 'persona_adherence.png'
        plot_persona_adherence(
            eval_dict=persona_eval_dict,
            alpha_persona_name=alpha_name,
            beta_persona_name=beta_name,
            save_path=str(persona_plot_path),
            show_plot=show_plots,
        )
        print(f"[Persona Evaluation] Saved plot to: {persona_plot_path}")
    
    # === Save data ===
    eval_data = {
        'metadata': {
            'scenario_id': scenario_id, 'question_id': question_id, 'question_variant': question_variant,
            'side_order': side_order, 'debate_arena': debate_arena,
            'alpha_persona_id': alpha_persona_id, 'alpha_persona_name': alpha_name,
            'beta_persona_id': beta_persona_id, 'beta_persona_name': beta_name,
        },
        'intra_agent_honesty': intra_scores,
        'inter_agent_alignment': {'external': inter_external, 'internal': inter_internal},
        'persona_adherence': persona_eval_dict,  # NEW: Add persona evaluation
    }
    with open(eval_output_dir / 'eval_data.json', 'w') as f:
        json.dump(eval_data, f, indent=2)
    
    if snapshot_path.exists():
        shutil.copy(snapshot_path, eval_output_dir / 'debate_snapshot.json')
    
    print(f"\n✓ Saved to: {eval_output_dir}/")
    return agora, analyzer, persona_eval_dict

In [5]:
# ===== RUN ALL EXPERIMENTS =====
import itertools
from datetime import datetime

# Get all scenarios and their questions
all_scenarios = debate_construction.get("pair_scenarios", [])
variants = ["controversial"] #["agreeable", "controversial"]
side_orders = ["12"]#["12", "21"]
arenas = ["bias"]#["neutral", "bias"]

# Settings
TURNS_PER_AGENT = 4
VERBOSE = False  # Set True to see full debate output

# Build all combinations
# Filter to specific scenarios (or set to None to run all)
SELECTED_SCENARIOS = ["cold_war_deterrence"] #["peer_collab_1"]  # e.g., ["peer_collab_1", "advers_hostile_2"] or None for all

combinations = []
for scenario in all_scenarios:
    scenario_id = scenario["id"]
    if SELECTED_SCENARIOS is None or scenario_id in SELECTED_SCENARIOS:
        # Each scenario has one question_id (not question_ids)
        for variant, side, arena in itertools.product(variants, side_orders, arenas):
            combinations.append((scenario_id, 0, variant, side, arena))

print(f"Total experiments: {len(combinations)}")
print(f"Scenarios: {[s['id'] for s in all_scenarios]}")
print(f"Variants: {variants}, Side orders: {side_orders}, Arenas: {arenas}")
print(f"\n{'='*60}")

# Run all experiments
results = []
start_time = datetime.now()

for i, (scenario_id, q_idx, variant, side, arena) in enumerate(combinations, 1):
    print(f"\n[{i}/{len(combinations)}] {scenario_id} q{q_idx} {variant} {side} {arena}")
    try:
        agora, analyzer = run_experiment(
            scenario_id=scenario_id,
            question_index=q_idx,
            question_variant=variant,
            side_order=side,
            debate_arena=arena,
            turns_per_agent=TURNS_PER_AGENT,
            verbose=VERBOSE,
            show_plots=False,  # Don't display plots in batch mode
            evaluate_persona=True,  # Enable persona evaluation
            persona_eval_verbose=True,  # Show evaluation progress
        )
        results.append({"combo": (scenario_id, q_idx, variant, side, arena), "success": True})
    except Exception as e:
        print(f"  ✗ FAILED: {e}")
        results.append({"combo": (scenario_id, q_idx, variant, side, arena), "success": False, "error": str(e)})

# Summary
duration = datetime.now() - start_time
print(f"\n{'='*60}")
print(f"COMPLETE! {sum(r['success'] for r in results)}/{len(results)} succeeded")
print(f"Total time: {duration}")


Total experiments: 1
Scenarios: ['peer_collab_1', 'peer_collab_2', 'hier_account_1', 'hier_account_2', 'struct_comp_1', 'struct_comp_2', 'advers_hostile_1', 'advers_hostile_2', 'family_privacy', 'family_finance', 'cold_war_deterrence', 'cold_war_information', 'self_budget_officer', 'self_small_business_owner', 'self_unionized_worker', 'self_parent', 'self_us_envoy', 'self_ussr_envoy']
Variants: ['controversial'], Side orders: ['12'], Arenas: ['bias']


[1/1] cold_war_deterrence q0 controversial 12 bias
[run] scenario=cold_war_deterrence  q=cold_war_deterrence_q/controversial  alpha=us_cold_war_envoy  beta=ussr_cold_war_envoy  arena=bias (us_cold_war_envoy)


/home/snoroozi/anaconda3/envs/agora_eval/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Loading model: all-mpnet-base-v2...


/home/snoroozi/anaconda3/envs/agora_eval/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



[Persona Evaluation] Running persona adherence evaluation...
Evaluating Alpha against persona: us_cold_war_envoy
  Turn 1: Scoring individual public speech...
  Turn 1: Scoring individual private reflection...
  Turn 1: Scoring cumulative public (turns 1-1)...
  Turn 1: Scoring cumulative private (turns 1-1)...
  Turn 2: Scoring individual public speech...
  Turn 2: Scoring individual private reflection...
  Turn 2: Scoring cumulative public (turns 1-2)...
  Turn 2: Scoring cumulative private (turns 1-2)...
  Turn 3: Scoring individual public speech...
  Turn 3: Scoring individual private reflection...
  Turn 3: Scoring cumulative public (turns 1-3)...
  Turn 3: Scoring cumulative private (turns 1-3)...
  Turn 4: Scoring individual public speech...
  Turn 4: Scoring individual private reflection...
  Turn 4: Scoring cumulative public (turns 1-4)...
  Turn 4: Scoring cumulative private (turns 1-4)...

Evaluating Beta against persona: ussr_cold_war_envoy
  Turn 1: Scoring individual pub